In [1]:
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Czyszczenie potencjalnie uszkodzonych pakietów i instalacja stabilnych wersji
!pip uninstall -y xformers trl peft accelerate bitsandbytes unsloth
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --prefer-binary xformers trl peft accelerate bitsandbytes

Found existing installation: peft 0.21.1
Uninstalling peft-0.21.1:
  Successfully uninstalled peft-0.21.1
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-77260ke5/unsloth_52f9bb83a6234c9184e5a6155946208e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-77260ke5/unsloth_52f9bb83a6234c9184e5a6155946208e
  Resolved https://github.com/unslothai/unsloth.git to commit 6c159816c5eec6639c1387fc02611362929f7e30
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 342.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 312.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 262.5 MB/s eta 0:00:00
   ━━━━━

In [3]:
from huggingface_hub import login
login('my_huggingface_token')

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'meta-llama/Meta-Llama-3-8B-Instruct',
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.6: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.10.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [6]:
data = {
    'train': '/content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Data/data_train.jsonl',
    'val': '/content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Data/data_val.jsonl',
    'test': '/content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Data/data_test.jsonl'
}

In [8]:
dataset = load_dataset("json", data_files=data)

In [9]:
def format_prompts(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []

    for instruction, input_text, output in zip(instructions, inputs, outputs):
        if input_text and input_text.strip() != "":
            user_content = f"{instruction}\n\nArticle / Context:\n{input_text}"
        else:
            user_content = instruction

        messages = [
            {"role": "system", "content": "You are a precise clinical decision-support assistant. Rely strictly on the provided medical context."},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output}
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)

    return { "text" : texts }

dataset = dataset.map(format_prompts, batched=True)
print(dataset["train"][0]["text"])

Map:   0%|          | 0/257 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a precise clinical decision-support assistant. Rely strictly on the provided medical context.<|eot_id|><|start_header_id|>user<|end_header_id|>

You are a health educator discussing mental health in hEDS. Based on the article, why is psychological wellbeing important and how can negative healthcare experiences affect patients?

Article / Context:
Psychological dysfunction and emotional problems, including depression, anxiety, affective disorder, low self-confidence, negative thinking, hopelessness, and desperation, are also common among those with EDS. These problems may exacerbate the pain experience, as well as other organ system manifestations (especially gastrointestinal and autonomic). This can lead to avoidance behavior, exacerbation of dysfunction and disability, and marginalization. To be ignored, being assigned a psychological and/or psychiatric explanation, not being respected and treated as an object could 

In [12]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["val"],
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 50,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Models",
        report_to = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/257 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/32 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/257 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 257 | Num Epochs = 2 | Total steps = 66
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss,Validation Loss
50,1.008657,1.098163
66,0.947341,1.074238


Filter:   0%|          | 0/32 [00:00<?, ? examples/s]

Unsloth: Not an error, but `num_items_in_batch` could not be counted for LlamaForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Models/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Models/checkpoint-66/tokenizer_config.json.


In [15]:
import pandas as pd
from tqdm import tqdm
import torch

FastLanguageModel.for_inference(model)

results = []

print("Generate results for training data!")
for rec in tqdm(dataset["test"]):
    if rec["input"] and rec["input"].strip() != "":
        user_content = f"{rec['instruction']}\n\nArticle / Context:\n{rec['input']}"
    else:
        user_content = rec["instruction"]

    messages = [
        {"role": "system", "content": "You are a precise clinical decision-support assistant. Rely strictly on the provided medical context."},
        {"role": "user", "content": user_content},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids = inputs,
            max_new_tokens = 512,
            temperature = 0.1,
            do_sample = True,
            use_cache = True,
        )

    pred = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens = True)

    results.append({
        "instruction": rec["instruction"],
        "input": rec["input"],
        "expected_output": rec["output"],
        "model_prediction": pred
    })


results_df = pd.DataFrame(results)

csv_path = "/content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Results/test_results_llama.csv"
jsonl_path = "/content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Results/test_results_llama.jsonl"

results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
results_df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)

print(f"Ready!\nSaved {len(results_df)} tested records in:\n- {csv_path}\n- {jsonl_path}")

Generate results for training data!


  0%|          | 0/32 [00:00<?, ?it/s]Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
100%|██████████| 32/32 [09:59<00:00, 18.73s/it]

Ready!
Saved 32 tested records in:
- /content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Results/test_results_llama.csv
- /content/drive/MyDrive/GenAI/LLM/Fine-tuned LLM/hEDS/Results/test_results_llama.jsonl
